### Chapter 4.2 - The Image Classification Dataset

Image classification uses images as inputs and class IDs as labels. This notebook uses tiny synthetic image tensors so the data shape and batching ideas are visible without downloading a dataset.


In [ ]:
import math
import random
import numpy as np
import torch


### 4.2.1 Loading the Dataset

#### 1. Intuition

An image dataset contains image tensors and labels. A grayscale image can be stored as a matrix of pixel values.

A pixel is one small measurement in an image. A channel is one layer of pixel measurements, such as grayscale, red, green, or blue.

#### 2. Why this exists

Image models need a consistent tensor shape. For many PyTorch image datasets, the shape is `(examples, channels, height, width)`.


#### 3. Examples

Create a tiny image dataset with four grayscale images.


In [ ]:
images = torch.arange(4 * 1 * 2 * 2, dtype=torch.float32)
images = images.reshape(4, 1, 2, 2)
labels = torch.tensor([0, 1, 0, 1])
dataset = torch.utils.data.TensorDataset(images, labels)
len(dataset), images.shape, labels.shape


#### 4. Step-by-step breakdown

`torch.arange(...)` creates enough pixel values for four small images.

`reshape(4, 1, 2, 2)` means 4 examples, 1 channel, height 2, width 2.

`labels` stores one class ID per image.

`TensorDataset(images, labels)` keeps each image aligned with its label.

#### 5. Connection to ML systems

Real datasets such as Fashion-MNIST follow the same idea but with many more images and larger spatial dimensions.

#### 6. Common confusion points

- Image tensors have shape conventions; always check them.
- Labels should align with the first dimension of images.
- Pixel values are numbers, even when the image looks visual to humans.
- Loading data is separate from training a model.


### 4.2.2 Reading a Minibatch

#### 1. Intuition

A minibatch is a small group of examples returned together for one training step.

For images, a minibatch keeps the image dimensions and adds or preserves the batch dimension.

#### 2. Why this exists

Training one image at a time wastes hardware efficiency. Training all images at once may use too much memory. Minibatches balance both concerns.


#### 3. Examples

Read one minibatch using a DataLoader.


In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)
batch_images, batch_labels = next(iter(loader))
batch_images.shape, batch_labels.shape


Flatten images when using a linear classifier.


In [ ]:
flat = batch_images.reshape(batch_images.shape[0], -1)
flat.shape


#### 4. Step-by-step breakdown

`DataLoader` groups dataset items into batches.

`next(iter(loader))` asks for the first batch.

`batch_images.shape` is `(2, 1, 2, 2)` for two tiny grayscale images.

`reshape(batch_images.shape[0], -1)` keeps the batch size and flattens each image into one feature vector.

#### 5. Connection to ML systems

Softmax regression is a linear classifier, so it expects feature vectors. For images, that often means flattening pixels before the linear layer.

#### 6. Common confusion points

- Flattening removes spatial layout information.
- `-1` asks PyTorch to infer the remaining dimension.
- The batch dimension should stay first.
- Shuffling should not break image-label pairing.


### 4.2.3 Visualization

#### 1. Intuition

Visualization means displaying data so humans can inspect it.

For images, visual inspection helps catch wrong labels, wrong shapes, inverted colors, or normalization mistakes.

#### 2. Why this exists

Models only see numbers, but humans often need visual checks to verify the data pipeline.


#### 3. Examples

Prepare one tiny image for display as a 2D grid.


In [ ]:
image = images[0]
grid = image.squeeze(0)
label = labels[0]
grid, label


A class-name lookup turns label IDs into readable names.


In [ ]:
class_names = ["dark", "bright"]
name = class_names[int(label)]
name


#### 4. Step-by-step breakdown

`images[0]` selects the first image.

`squeeze(0)` removes the channel dimension when it has size 1.

The label is an integer class ID.

`class_names[int(label)]` maps the class ID to a human-readable name.

#### 5. Connection to ML systems

Dataset visualization is usually done before training. It is a cheap way to find data mistakes that no optimizer can fix.

#### 6. Common confusion points

- Visualization is for inspection, not training.
- A label ID needs a class-name mapping to be readable.
- Squeezing removes dimensions only when their size is 1.
- Do not confuse display shape with model input shape.


### 4.2.4 Summary

#### 1. Intuition

Image classification datasets pair image tensors with class labels.

The key shape convention is usually `(batch, channels, height, width)`.

#### 2. Why this exists

Correct data shape and label alignment are prerequisites for meaningful classification training.


#### 3. Examples

Summarize one batch's structure.


In [ ]:
summary = {
    "batch_size": batch_images.shape[0],
    "channels": batch_images.shape[1],
    "height": batch_images.shape[2],
    "width": batch_images.shape[3],
}
summary


#### 4. Step-by-step breakdown

The summary names each dimension.

Naming dimensions reduces confusion when tensors become larger.

The label tensor stores one class ID per image in the batch.

#### 5. Connection to ML systems

Future image models will preserve spatial layout longer, but softmax regression starts by flattening images into feature vectors.

#### 6. Common confusion points

- Always inspect batch shapes.
- Always preserve image-label alignment.
- Flattening is convenient but loses spatial structure.
- Class labels are usually integer IDs in PyTorch classification.


### 4.2.5 Exercises

#### 1. Intuition

These exercises practice reading and reshaping image batches.

#### 2. Why this exists

Most image-classification bugs start as data-shape bugs.


#### 3. Examples

Exercise 1: create a batch of three 4 by 4 grayscale images.


In [ ]:
X = torch.zeros(3, 1, 4, 4)
y = torch.tensor([0, 1, 2])
X.shape, y.shape


Exercise 2: flatten the image batch for a linear classifier.


In [ ]:
flat = X.reshape(X.shape[0], -1)
flat.shape


#### 4. Step-by-step breakdown

Exercise 1 checks the image batch shape convention.

Exercise 2 checks flattening while preserving the batch dimension.

The flattened feature count is `channels * height * width`.

#### 5. Connection to ML systems

Flattened image batches feed directly into softmax regression and multilayer perceptrons.

#### 6. Common confusion points

- The batch dimension is not an image dimension.
- Flatten only the dimensions that belong to each example.
- The number of labels should equal the batch size.
- The class ID range should match the number of classes.
